# AeroCaliper: Vertex AI Search RAG Deep Dive
## How Policy-Grounded AI Governance Works — An Interactive Tutorial

---

### The Problem: Hardcoded Rules Don't Scale

Every enterprise has policies — FinOps cost controls, HR privacy rules, security mandates. The naive approach is to hardcode these rules directly into an AI agent's system prompt:

```
You MUST use spot instances for batch jobs.
You MUST NOT share salary data with unauthorized users.
...(200 more rules)...
```

This **breaks immediately** when:
- Policy changes (requires a code deploy)
- Policy grows (token limits get hit)
- You have multiple departments (you can't cram everything in)

### The AeroCaliper Solution: Policy-as-a-Service via RAG

AeroCaliper treats enterprise policy documents as **living data**, not static code:

1. **Upload** PDF/TXT policy documents to a GCS bucket (one per domain)
2. **Index** them with Vertex AI Search (Agent Builder)
3. **Query** at runtime — fetch only the *relevant* policy clause for the current violation
4. **Inject** that grounded context into the LLM prompt

The result: policy updates are instant, departments are isolated, and token usage stays minimal.

---
## Section 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "aerocaliper")
LOCATION = os.getenv("VERTEX_SEARCH_LOCATION", "global")
FINOPS_DATASTORE_ID = os.getenv("VERTEX_DATASTORE_ID_FINOPS", "finops-ds")
HR_DATASTORE_ID = os.getenv("VERTEX_DATASTORE_ID_HR", "hr-ds")

print(f"Project : {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"FinOps DS: {FINOPS_DATASTORE_ID}")
print(f"HR DS    : {HR_DATASTORE_ID}")

In [ ]:
from google.cloud import discoveryengine_v1

# Initialize the search client — uses Application Default Credentials
client = discoveryengine_v1.SearchServiceClient()
print("✅ Vertex AI Search client initialized")

---
## Section 2: Standard Vector Search vs. Extractive Answers

Most RAG tutorials show you how to get back a **full document chunk** from a vector search. That gives you something like:

> *"3.4 Cloud Resource Governance. All workloads submitted to the enterprise compute fabric MUST be assigned a cost center code prior to provisioning. Section 3.4.1 defines cost center hierarchy..."*

You then have to feed that entire wall of text to the LLM, burning tokens and diluting the signal.

**Vertex AI's `extractive_answers`** is different. It uses a reading-comprehension model internally to extract the **single most relevant sentence or clause** from the policy document that answers your query — like a pre-trained QA step baked into the search layer.

The difference in practice:

| Approach | Result |
|---|---|
| Standard vector chunk | 500 tokens of tangentially relevant text |
| `extractive_answers` | `"For batch training workloads, use_spot MUST be set to true."` |

This is why AeroCaliper's Phase 3 uses `extractive_answers` exclusively.

---
## Section 3: Querying the FinOps Policy Datastore

In [ ]:
# Build the serving config path — this identifies the specific datastore + engine
finops_serving_config = client.serving_config_path(
    PROJECT_ID, LOCATION, FINOPS_DATASTORE_ID, "default_config"
)

finops_query = "FinOps Routing Policy Spot Instances Budget Tag"

finops_request = discoveryengine_v1.SearchRequest(
    serving_config=finops_serving_config,
    query=finops_query,
    page_size=1,
)

print(f"Query: '{finops_query}'")
print(f"Targeting datastore: {FINOPS_DATASTORE_ID}")

---
## Section 4: Parsing the Response — Extractive Answers

In [ ]:
finops_response = client.search(finops_request)
print(f"Results returned: {len(finops_response.results)}")

In [ ]:
finops_snippets = []

for result in finops_response.results:
    derived = result.document.derived_struct_data
    extractive_answers = derived.get("extractive_answers", [])
    
    for answer in extractive_answers:
        content = answer.get("content", "")
        if content:
            finops_snippets.append(content)

if finops_snippets:
    print("\n✅ FinOps Policy — Extractive Answer Retrieved:")
    print("=" * 60)
    for snippet in finops_snippets:
        print(snippet)
else:
    print("⚠️  No extractive answers found.")
    print("   This usually means the datastore index is still propagating (10-30 min after import).")
    print("   Or: extractive content feature needs to be enabled on the datastore.")

---
## Section 5: The 'Universal' Proof — Swapping to HR Privacy Policy

Here is where AeroCaliper's multi-tenant architecture shines.

The **exact same code** — zero changes to logic — queries a completely isolated HR policy datastore. No FinOps rules contaminate the HR query. No HR rules bleed into FinOps.

Each department owns its own:
- GCS bucket (`aerocaliper-rag-finops` / `aerocaliper-rag-hr`)
- Vertex AI Search datastore (`finops-ds` / `hr-ds`)
- Golden dataset rows (`golden_dataset.csv` rows filtered by domain)

This is strict **policy isolation** — a mandatory requirement for any enterprise AI governance platform.

In [ ]:
# Identical code — just swap the datastore ID and query
hr_serving_config = client.serving_config_path(
    PROJECT_ID, LOCATION, HR_DATASTORE_ID, "default_config"
)

hr_query = "HR Privacy PII Salary Restrictions"

hr_request = discoveryengine_v1.SearchRequest(
    serving_config=hr_serving_config,
    query=hr_query,
    page_size=1,
)

hr_response = client.search(hr_request)
hr_snippets = []

for result in hr_response.results:
    derived = result.document.derived_struct_data
    for answer in derived.get("extractive_answers", []):
        content = answer.get("content", "")
        if content:
            hr_snippets.append(content)

if hr_snippets:
    print("\n✅ HR Privacy Policy — Extractive Answer Retrieved:")
    print("=" * 60)
    for snippet in hr_snippets:
        print(snippet)
else:
    print("⚠️  No extractive answers found for HR datastore.")
    print("   Index may still be propagating.")

---
## How This Fits Into AeroCaliper's Phase 3

This exact retrieval pattern — `SearchRequest` → `extractive_answers` → snippet injection — is what runs in `aerocaliper.py` inside `diagnostic_phase()`:

```python
# aerocaliper.py Phase 3 (simplified)
response = client.search(request)
snippets = []
for result in response.results:
    for ext in result.document.derived_struct_data.get("extractive_answers", []):
        snippets.append(ext.get("content", ""))

if snippets:
    retrieved_policy = "\n".join(snippets)
else:
    raise RuntimeError("Datastore indexing in progress.")
```

The retrieved policy snippet is then injected directly into the Gemini diagnostic prompt, grounding the LLM in real enterprise policy before it generates the remediation patch.

**No hallucination. No hardcoding. No token bloat.** Just the exact sentence from the policy that applies to the current violation.